House Price Prediction using Artificial Intelligence
### AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026

Student: Pandhula Sinduja 
Roll No.: 160723733125
Branch: CSE  
College: Methodist College of Engineering and Technology

> End-to-end data analytics and Artificial Intelligence project for predicting residential house sale prices.

## HOUSE PRICE PREDICTION - CODE FILE

# Files:
1. House_Price_Prediction_IBM_Internship.ipynb
   Complete Jupyter Notebook implementation.

2. requirements.txt
   Python libraries required to run the notebook.

# Run:
    pip install -r requirements.txt
    jupyter notebook

# Then open:
    House_Price_Prediction_IBM_Internship.ipynb

# Run all cells using:
    Kernel -> Restart & Run All


## 1. Project Objective
Build a regression model that learns from house characteristics and predicts `SalePrice`.

**Pipeline:** Data Loading → EDA → Cleaning → Encoding → Train/Validation Split → Model Training → Evaluation → Prediction

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42


## 2. Load the House Prediction Dataset

The notebook first checks for a local `train.csv`. If it is not present, it attempts to load a public CSV mirror of the Kaggle House Prices dataset.

**Official source:** Kaggle – House Prices: Advanced Regression Techniques.

In [ ]:
from pathlib import Path

LOCAL_FILE = Path("train.csv")
DATA_URL = "https://raw.githubusercontent.com/ankita1112/House-Prices-Advanced-Regression/master/train.csv"

if LOCAL_FILE.exists():
    df = pd.read_csv(LOCAL_FILE)
    source_used = "Local train.csv"
else:
    df = pd.read_csv(DATA_URL)
    source_used = "Public CSV mirror"

print("Data source:", source_used)
print("Dataset shape:", df.shape)
display(df.head())


## 3. Data Understanding

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("\nData types:")
display(df.dtypes.value_counts())

print("\nSummary statistics:")
display(df.describe(include="all").T.head(20))


## 4. Missing-Value Analysis

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("Columns containing missing values:", len(missing))
display(missing.head(20).to_frame("Missing Values"))

plt.figure(figsize=(10, 6))
missing.head(20).sort_values().plot(kind="barh")
plt.title("Top 20 Features with Missing Values")
plt.xlabel("Number of Missing Values")
plt.tight_layout()
plt.show()


## 5. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of House Sale Prices")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=0.65)
plt.title("Above-Ground Living Area vs Sale Price")
plt.xlabel("Living Area (sq ft)")
plt.ylabel("Sale Price")
plt.tight_layout()
plt.show()


In [ ]:
numeric_cols_for_corr = df.select_dtypes(include=np.number).columns
corr = df[numeric_cols_for_corr].corr(numeric_only=True)["SalePrice"].sort_values(ascending=False)

print("Top numerical correlations with SalePrice:")
display(corr.head(12).to_frame("Correlation"))

top_corr_cols = corr.head(10).index
plt.figure(figsize=(10, 7))
sns.heatmap(df[top_corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Key Numerical Features")
plt.tight_layout()
plt.show()


## 6. Feature Preparation

- `SalePrice` is the target.
- Numerical features use median imputation.
- Categorical features use most-frequent imputation followed by one-hot encoding.
- `log1p(SalePrice)` is used during training to make the skewed target easier to model.


In [ ]:
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

# ID is an identifier rather than a useful predictive feature.
if "Id" in X.columns:
    X = X.drop(columns=["Id"])

y_log = np.log1p(y)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y_log, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))


## 7. Train Multiple Regression Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=250,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=250,
        learning_rate=0.04,
        max_depth=3,
        random_state=RANDOM_STATE
    )
}

results = []
fitted_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pred_log = pipe.predict(X_valid)

    actual = np.expm1(y_valid)
    predicted = np.maximum(0, np.expm1(pred_log))

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    rmsle = np.sqrt(mean_squared_error(np.log1p(actual), np.log1p(predicted)))

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "RMSLE": rmsle
    })
    fitted_pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)


## 8. Model Comparison

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=results_df, x="Model", y="RMSE")
plt.title("Model Comparison – Validation RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE (lower is better)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_pipe = fitted_pipelines[best_model_name]

print("Best validation model by RMSE:", best_model_name)


## 9. Actual vs Predicted Prices

In [ ]:
best_pred_log = best_pipe.predict(X_valid)
actual = np.expm1(y_valid)
predicted = np.maximum(0, np.expm1(best_pred_log))

plt.figure(figsize=(8, 8))
sns.scatterplot(x=actual, y=predicted, alpha=0.65)
low = min(actual.min(), predicted.min())
high = max(actual.max(), predicted.max())
plt.plot([low, high], [low, high], linestyle="--")
plt.title(f"Actual vs Predicted Sale Prices – {best_model_name}")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.tight_layout()
plt.show()


## 10. Feature Importance

In [ ]:
if best_model_name in ["Random Forest", "Gradient Boosting"]:
    fitted_preprocessor = best_pipe.named_steps["preprocessor"]
    fitted_model = best_pipe.named_steps["model"]

    feature_names = fitted_preprocessor.get_feature_names_out()
    importances = pd.Series(
        fitted_model.feature_importances_,
        index=feature_names
    ).sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 7))
    importances.sort_values().plot(kind="barh")
    plt.title(f"Top 15 Feature Importances – {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()

    display(importances.to_frame("Importance"))
else:
    print("Feature importance is shown for tree-based models.")


## 11. Example House Price Predictions

In [ ]:
sample = X_valid.head(5).copy()
sample_pred = np.maximum(0, np.expm1(best_pipe.predict(sample)))

prediction_table = pd.DataFrame({
    "House_Index": sample.index,
    "Predicted_SalePrice": np.round(sample_pred, 2)
}).reset_index(drop=True)

display(prediction_table)


## 12. Conclusion

This project demonstrates a complete real-world machine-learning workflow for housing-price analytics. The final notebook:

- audits the dataset,
- performs exploratory analysis,
- handles missing values,
- encodes categorical data,
- trains multiple regression models,
- evaluates predictions with standard regression metrics,
- visualizes model behavior, and
- produces price estimates for unseen houses.

### Future Improvements
- Cross-validation and hyperparameter tuning
- XGBoost / LightGBM / CatBoost comparison
- SHAP-based explainability
- Streamlit dashboard
- Cloud deployment and model monitoring